# ONIA · 5.2 Încărcarea și explorarea corpusurilor

Notebook de lucru pentru liceu. Datele sunt sintetice și fișierele sunt distribuite în directorul `datasets` al site-ului.

**Rulare:** pornește JupyterLab din rădăcina proiectului pentru a păstra căile din exemple. Dacă ai descărcat doar notebookul, descarcă și dataseturile în `datasets/`.

## Exemplul 1. Citim un TXT și numărăm liniile goale

Schimbă datele, execută din nou și explică de ce s-au modificat rezultatele.

In [ ]:
from pathlib import Path
fisier = Path("datasets/nlp-corpus-documente.txt")
with fisier.open("r", encoding="utf-8") as f:
    linii = f.readlines()

texte = [linie.strip() for linie in linii]
goale = sum(text == "" for text in texte)
documente = [text for text in texte if text]
print("Linii:", len(linii), "Goale:", goale)
print("Documente valide:", documente)


## Exemplul 3. CSV din biblioteca standard (ghilimele corecte)

Schimbă datele, execută din nou și explică de ce s-au modificat rezultatele.

In [ ]:
import csv
from io import StringIO

continut = 'id,text,eticheta\n001,"Felicitări, ai câștigat!",spam\n002,"Salut, Ana!",legitim\n'
f = StringIO(continut)
for rand in csv.DictReader(f):
    print(rand["id"], rand["text"], rand["eticheta"])


## Exemplul 4. Citire Pandas cu tipuri explicite

Schimbă datele, execută din nou și explică de ce s-au modificat rezultatele.

In [ ]:
import pandas as pd

df = pd.read_csv(
    "datasets/nlp-corpus-train.csv",
    encoding="utf-8",
    dtype={"id": "string", "text": "string", "eticheta": "string"},
    usecols=["id", "text", "eticheta", "sursa"]
)
print(df.head(4).to_string(index=False))
print("Dimensiune:", df.shape)
print("Tipuri:", df.dtypes.to_dict())


## Exemplul 5. JSON clasic și câmpuri nested

Schimbă datele, execută din nou și explică de ce s-au modificat rezultatele.

In [ ]:
import json
import pandas as pd

raw = '[{"id": "01", "text": "Meci de tenis", "meta": {"sursa": "redactie-A"}}, {"id": "02", "text": "Cod Python", "meta": {"sursa": "redactie-B"}}]'
documente = json.loads(raw)
df = pd.json_normalize(documente)
print(df[["id", "text", "meta.sursa"]].to_string(index=False))


## Exemplul 6. Citim câte un document JSON pe linie

Schimbă datele, execută din nou și explică de ce s-au modificat rezultatele.

In [ ]:
import json
from pathlib import Path

docuri = []
cale = Path("datasets/nlp-corpus-documente.jsonl")
with cale.open(encoding="utf-8") as f:
    for numar_linie, linie in enumerate(f, start=1):
        if not linie.strip():
            continue
        obiect = json.loads(linie)
        docuri.append((obiect["id"], obiect["text"]))
print("Documente citite:", len(docuri))
print("Primul:", docuri[0])


## Exemplul 7. Auditul unui corpus

Schimbă datele, execută din nou și explică de ce s-au modificat rezultatele.

In [ ]:
import pandas as pd

df = pd.read_csv("datasets/nlp-corpus-train.csv", dtype={"id": "string"})
print("Rânduri/coloane:", df.shape)
print("Coloane:", df.columns.tolist())
print("Lipsuri pe coloană:\n", df.isna().sum())
print("ID-uri unice:", df["id"].is_unique)
print("Etichete:\n", df["eticheta"].value_counts())
print("Primele 3:\n", df.head(3).to_string(index=False))


## Exemplul 8. Politică explicită pentru documente vide

Schimbă datele, execută din nou și explică de ce s-au modificat rezultatele.

In [ ]:
import pandas as pd

df = pd.read_csv("datasets/nlp-corpus-train.csv")
serie = df["text"].astype("string")
lipsa = serie.isna()
gol = serie.fillna("").str.strip().eq("")
print("Celule lipsă:", int(lipsa.sum()))
print("Documente vide sau doar spații:", int(gol.sum()))

curat = df.loc[~gol].copy()
print("Documente păstrate:", len(curat))


## Exemplul 9. Construim o cheie normalizată de deduplicare

Schimbă datele, execută din nou și explică de ce s-au modificat rezultatele.

In [ ]:
import pandas as pd

df = pd.read_csv("datasets/nlp-corpus-train.csv")
text = df["text"].astype("string")
df["cheie"] = text.fillna("").str.casefold().str.split().str.join(" ")
print("Duplicate după normalizare:", int(df.duplicated("cheie").sum()))

conflicte = df.groupby("cheie")["eticheta"].nunique()
print("Chei cu etichete contradictorii:", (conflicte > 1).sum())
print(conflicte[conflicte > 1])


## Exemplul 10. Eliminăm doar duplicatele necontradictorii

Schimbă datele, execută din nou și explică de ce s-au modificat rezultatele.

In [ ]:
import pandas as pd

df = pd.read_csv("datasets/nlp-corpus-train.csv")
df["cheie"] = df["text"].fillna("").str.casefold().str.split().str.join(" ")
df = df.loc[df["cheie"].ne("")].copy()

# Întâi raportăm contradicțiile, nu alegem arbitrar o etichetă.
numar_etichete = df.groupby("cheie")["eticheta"].transform("nunique")
conflict = numar_etichete > 1
print("Rânduri de verificat manual:", int(conflict.sum()))

# Doar pentru exemplul de antrenare scoatem conflictele din datele curate.
df_curat = df.loc[~conflict].drop_duplicates("cheie", keep="first")
print("Rânduri curate:", len(df_curat))


## Exemplul 11. Statistici de lungime cu Pandas

Schimbă datele, execută din nou și explică de ce s-au modificat rezultatele.

In [ ]:
import pandas as pd

df = pd.read_csv("datasets/nlp-corpus-train.csv")
texte = df["text"].fillna("")
df["caractere"] = texte.str.len()
df["cuvinte"] = texte.str.split().str.len()
print(df["cuvinte"].describe(percentiles=[.5, .75, .95]))
print(df.groupby("eticheta")["cuvinte"].agg(["count", "mean", "median"]))


## Exemplul 12. Termeni frecvenți și DF, de la zero

Schimbă datele, execută din nou și explică de ce s-au modificat rezultatele.

In [ ]:
from collections import Counter
import re

docs = ["python date python", "sport și tenis", "python algoritm"]
tokeni = [re.findall(r"\w+", text.casefold()) for text in docs]
TF = Counter(t for doc in tokeni for t in doc)
DF = Counter(t for doc in tokeni for t in set(doc))
print("TF(python):", TF["python"])
print("DF(python):", DF["python"])
print("Vocabular:", sorted(TF))


## Exemplul 13. Etichete, proporții și surse

Schimbă datele, execută din nou și explică de ce s-au modificat rezultatele.

In [ ]:
import pandas as pd

df = pd.read_csv("datasets/nlp-corpus-train.csv")
print(df["eticheta"].value_counts())
print(df["eticheta"].value_counts(normalize=True).round(3))
print(pd.crosstab(df["sursa"], df["eticheta"]))


## Exemplul 14. Stratificare fără contaminarea vocabularului

Schimbă datele, execută din nou și explică de ce s-au modificat rezultatele.

In [ ]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report

df = pd.read_csv("datasets/nlp-corpus-train.csv")
df = df.loc[df["text"].notna()].copy()
df["cheie"] = df["text"].str.casefold().str.split().str.join(" ")
# Verificăm conflictele de etichetare și scoatem temporar
# rândurile afectate din exemplul de antrenare.
numar_etichete = df.groupby("cheie")["eticheta"].transform("nunique")
df = df.loc[numar_etichete == 1].drop_duplicates("cheie").copy()

X_train, X_val, y_train, y_val = train_test_split(
    df["text"], df["eticheta"], test_size=0.25,
    stratify=df["eticheta"], random_state=17
)
model = Pipeline([
    ("tfidf", TfidfVectorizer(ngram_range=(1, 2))),
    ("clasificator", LogisticRegression(max_iter=1000))
])
model.fit(X_train, y_train)
print(classification_report(y_val, model.predict(X_val), zero_division=0))


## Exemplul 15. Împărțire după grup, nu după document

Schimbă datele, execută din nou și explică de ce s-au modificat rezultatele.

In [ ]:
import pandas as pd
from sklearn.model_selection import GroupShuffleSplit

df = pd.read_csv("datasets/nlp-corpus-train.csv")
df = df.loc[df["text"].notna()].copy()
splitter = GroupShuffleSplit(n_splits=1, test_size=0.33, random_state=17)
train_idx, val_idx = next(splitter.split(df, groups=df["sursa"]))
train = df.iloc[train_idx]
val = df.iloc[val_idx]
print("Surse train:", sorted(train["sursa"].unique()))
print("Surse val:", sorted(val["sursa"].unique()))
print("Suprapunere:", set(train["sursa"]) & set(val["sursa"]))


## Exemplul 16. Numărăm etichetele folosind chunksize

Schimbă datele, execută din nou și explică de ce s-au modificat rezultatele.

In [ ]:
from collections import Counter
import pandas as pd

frecvente = Counter()
total = 0
for bucata in pd.read_csv("datasets/nlp-corpus-train.csv", chunksize=6):
    frecvente.update(bucata["eticheta"].dropna().tolist())
    total += len(bucata)
print("Total rânduri:", total)
print("Etichete:", dict(frecvente))


## Exemplul 17. Export sigur al predicțiilor

Schimbă datele, execută din nou și explică de ce s-au modificat rezultatele.

In [ ]:
import pandas as pd

test = pd.read_csv("datasets/nlp-corpus-test.csv", dtype={"id": "string"})
# Predicții de exemplu, nu rezultate ale unui model evaluat.
predictii = ["sport", "tehnologie", "educatie"]
assert len(predictii) == len(test)
sub = pd.DataFrame({"id": test["id"], "eticheta": predictii})
sub.to_csv("datasets/nlp-corpus-submission-demo.csv", index=False, encoding="utf-8")
verificare = pd.read_csv("datasets/nlp-corpus-submission-demo.csv", dtype={"id": "string"})
assert verificare["id"].tolist() == test["id"].tolist()
assert verificare["eticheta"].notna().all()
print(verificare.to_string(index=False))
